# 📦 Orders Dataset — Cleaning & Feature Engineering
---
**Dataset:** `orders.csv`  
**Goal:** Handle missing values, fix data types, validate data, and engineer time-based and financial features.  
**Final output:** `orders_cleaned.csv`

## 1. Import Libraries

In [3]:
import pandas as pd
import numpy as np

## 2. Load Dataset

In [4]:
# Load the raw orders CSV file
df = pd.read_csv("orders.csv")

# Quick shape check
print("Shape:", df.shape)
df.head()

Shape: (20000, 19)


,order_id,customer_id,restaurant_id,order_date,order_time,city,cuisine,order_status,payment_method,order_amount,delivery_fee,packaging_fee,discount_amount,surge_fee,final_amount,estimated_delivery_time,actual_delivery_time,customer_rating,prep_time_minutes
0,1,814,7,2025-01-06,00:33:00,Pune,Biryani,Delivered,UPI,706,50,39,131,0,664,38,38,3.9,25
1,2,1087,306,2025-02-13,18:45:00,Delhi,Italian,Cancelled,Credit Card,1275,34,34,58,0,1285,42,84,3.6,13
2,3,860,279,2025-01-11,20:21:00,Pune,Desserts,Delivered,Debit Card,152,78,34,193,55,126,21,40,4.9,26
3,4,1331,45,2025-03-29,14:06:00,Hyderabad,Burger,Delivered,Cash on Delivery,1359,41,28,23,91,1496,24,24,3.7,20
4,5,4992,167,2025-02-27,13:05:00,Jaipur,South Indian,Cancelled,Credit Card,1530,68,28,229,47,1444,45,45,3.9,13


## 3. Explore the Data

In [5]:
df.describe()

,order_id,customer_id,restaurant_id,order_amount,delivery_fee,packaging_fee,discount_amount,surge_fee,final_amount,estimated_delivery_time,actual_delivery_time,customer_rating,prep_time_minutes
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,19800.000000,20000.000000
mean,10000.500000,2505.753800,250.048700,970.786400,54.995150,24.932900,159.928800,17.302350,908.088000,34.990850,43.443450,4.094586,27.673100
std,5773.647028,1446.691045,144.370509,477.362028,20.594813,8.941247,81.033702,33.580171,485.506709,8.927769,13.641701,0.619735,10.029211
min,1.000000,1.000000,1.000000,150.000000,20.000000,10.000000,20.000000,0.000000,-75.000000,20.000000,20.000000,2.000000,10.000000
25%,5000.750000,1246.750000,124.000000,554.000000,37.000000,17.000000,91.000000,0.000000,495.000000,27.000000,33.000000,3.700000,19.000000
50%,10000.500000,2497.000000,249.000000,972.000000,55.000000,25.000000,159.000000,0.000000,909.000000,35.000000,43.000000,4.100000,28.000000
75%,15000.250000,3764.000000,375.000000,1379.000000,73.000000,33.000000,230.000000,0.000000,1325.000000,43.000000,52.000000,4.600000,36.000000
max,20000.000000,5000.000000,500.000000,1800.000000,90.000000,40.000000,300.000000,120.000000,1984.000000,50.000000,94.000000,5.000000,45.000000


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   order_id                 20000 non-null  int64  
 1   customer_id              20000 non-null  int64  
 2   restaurant_id            20000 non-null  int64  
 3   order_date               20000 non-null  str    
 4   order_time               20000 non-null  str    
 5   city                     20000 non-null  str    
 6   cuisine                  20000 non-null  str    
 7   order_status             20000 non-null  str    
 8   payment_method           19800 non-null  str    
 9   order_amount             20000 non-null  int64  
 10  delivery_fee             20000 non-null  int64  
 11  packaging_fee            20000 non-null  int64  
 12  discount_amount          20000 non-null  int64  
 13  surge_fee                20000 non-null  int64  
 14  final_amount             20000 no

## 4. Check Duplicates & Missing Values

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df.isnull().sum()

order_id                     0
customer_id                  0
restaurant_id                0
order_date                   0
order_time                   0
city                         0
cuisine                      0
order_status                 0
payment_method             200
order_amount                 0
delivery_fee                 0
packaging_fee                0
discount_amount              0
surge_fee                    0
final_amount                 0
estimated_delivery_time      0
actual_delivery_time         0
customer_rating            200
prep_time_minutes            0
dtype: int64

## 5. Handle Missing Values

In [6]:
# Fill missing values with the most frequent payment method (mode)
df['payment_method'] = df['payment_method'].fillna(df['payment_method'].mode()[0])

In [7]:
# Fill missing ratings with median
df['customer_rating']=df['customer_rating'].fillna(df['customer_rating'].median())

In [8]:
# Confirm no nulls remain in these two columns
print(df[['payment_method', 'customer_rating']].isnull().sum())

payment_method     0
customer_rating    0
dtype: int64


## 6. Fix Data Types

In [9]:
# Convert order_date from string to datetime format
df['order_date'] = pd.to_datetime(df['order_date'])

In [10]:
# Convert order_time string to proper time object
df['order_time'] = pd.to_datetime(
    df['order_time'],
    format='%H:%M:%S'
).dt.time

In [11]:
# Verify the conversion
print("Updated dtypes:")
print(df[['order_date', 'order_time']].dtypes)

Updated dtypes:
order_date    datetime64[us]
order_time            object
dtype: object


## 7. Data Validation

In [12]:
# --- Order Amount must be positive ---
invalid_amt = df[df['order_amount'] <= 0]
print(f"Invalid order amounts (<=0): {len(invalid_amt)}")

Invalid order amounts (<=0): 0


In [13]:
# --- Customer Rating must be between 1 and 5 ---
invalid_rating = df[(df['customer_rating'] < 1) | (df['customer_rating'] > 5)]
print(f"Invalid ratings (out of 1-5 range): {len(invalid_rating)}")

Invalid ratings (out of 1-5 range): 0


In [14]:
# --- Check expected categories ---
print("\norder_status unique values :", df['order_status'].unique())
print("payment_method unique values:", df['payment_method'].unique())


order_status unique values : <StringArray>
['Delivered', 'Cancelled', 'Refunded']
Length: 3, dtype: str
payment_method unique values: <StringArray>
['UPI', 'Credit Card', 'Debit Card', 'Cash on Delivery', 'Wallet']
Length: 5, dtype: str


## 8. Feature Engineering

In [15]:
# Extract month name, day name, and hour from order datetime columns
df['order_month'] = df['order_date'].dt.month_name() 
df['order_day']  = df['order_date'].dt.day_name()

In [16]:
df['order_hour']  = pd.to_datetime(
    df['order_time'].astype(str), format='%H:%M:%S'
).dt.hour     

In [17]:
# Platform cost = 60% of delivery fee (platform's cut) + packaging fee
df['estimated_cost'] = (df['delivery_fee'] * 0.6) + df['packaging_fee']

In [18]:
# Profit = amount received from customer - platform's operating cost
df['profit'] = (df['final_amount'] - df['discount_amount'] - df['estimated_cost'])

In [19]:
# Binary flag for cancelled orders — useful for cancellation rate analysis
df['is_cancelled'] = np.where(df['order_status'] == 'Cancelled', 1, 0)
print("Cancellation breakdown:")
print(df['is_cancelled'].value_counts())

Cancellation breakdown:
is_cancelled
0    18050
1     1950
Name: count, dtype: int64


In [20]:
df.head()

,order_id,customer_id,restaurant_id,order_date,order_time,city,cuisine,order_status,payment_method,order_amount,...,estimated_delivery_time,actual_delivery_time,customer_rating,prep_time_minutes,order_month,order_day,order_hour,estimated_cost,profit,is_cancelled
0,1,814,7,2025-01-06,00:33:00,Pune,Biryani,Delivered,UPI,706,...,38,38,3.9,25,January,Monday,0,69.0,464.0,0
1,2,1087,306,2025-02-13,18:45:00,Delhi,Italian,Cancelled,Credit Card,1275,...,42,84,3.6,13,February,Thursday,18,54.4,1172.6,1
2,3,860,279,2025-01-11,20:21:00,Pune,Desserts,Delivered,Debit Card,152,...,21,40,4.9,26,January,Saturday,20,80.8,-147.8,0
3,4,1331,45,2025-03-29,14:06:00,Hyderabad,Burger,Delivered,Cash on Delivery,1359,...,24,24,3.7,20,March,Saturday,14,52.6,1420.4,0
4,5,4992,167,2025-02-27,13:05:00,Jaipur,South Indian,Cancelled,Credit Card,1530,...,45,45,3.9,13,February,Thursday,13,68.8,1146.2,1


In [21]:
# create Late Delivery column 
df['late_delivery'] = np.where(df['actual_delivery_time'] > df['estimated_delivery_time'], 1, 0)

## 9. Final Check & Export

In [22]:
# Final shape and null check before saving
print("Final Shape:", df.shape)
print("\nNull values after all cleaning steps:")
print(df.isnull().sum())
df.head()

Final Shape: (20000, 26)

Null values after all cleaning steps:
order_id                   0
customer_id                0
restaurant_id              0
order_date                 0
order_time                 0
city                       0
cuisine                    0
order_status               0
payment_method             0
order_amount               0
delivery_fee               0
packaging_fee              0
discount_amount            0
surge_fee                  0
final_amount               0
estimated_delivery_time    0
actual_delivery_time       0
customer_rating            0
prep_time_minutes          0
order_month                0
order_day                  0
order_hour                 0
estimated_cost             0
profit                     0
is_cancelled               0
late_delivery              0
dtype: int64


,order_id,customer_id,restaurant_id,order_date,order_time,city,cuisine,order_status,payment_method,order_amount,...,actual_delivery_time,customer_rating,prep_time_minutes,order_month,order_day,order_hour,estimated_cost,profit,is_cancelled,late_delivery
0,1,814,7,2025-01-06,00:33:00,Pune,Biryani,Delivered,UPI,706,...,38,3.9,25,January,Monday,0,69.0,464.0,0,0
1,2,1087,306,2025-02-13,18:45:00,Delhi,Italian,Cancelled,Credit Card,1275,...,84,3.6,13,February,Thursday,18,54.4,1172.6,1,1
2,3,860,279,2025-01-11,20:21:00,Pune,Desserts,Delivered,Debit Card,152,...,40,4.9,26,January,Saturday,20,80.8,-147.8,0,1
3,4,1331,45,2025-03-29,14:06:00,Hyderabad,Burger,Delivered,Cash on Delivery,1359,...,24,3.7,20,March,Saturday,14,52.6,1420.4,0,0
4,5,4992,167,2025-02-27,13:05:00,Jaipur,South Indian,Cancelled,Credit Card,1530,...,45,3.9,13,February,Thursday,13,68.8,1146.2,1,0


In [23]:
df.to_csv("orders_cleaned.csv", index=False)
print("✅ orders_cleaned.csv saved successfully!")

✅ orders_cleaned.csv saved successfully!
